Richard Davies · **Automated Data Visualisation for Policymaking** · PP434 · 2026/27

# Useful loop patterns

Four patterns you will use again and again:

1. **Building a list**: start with an empty list and add to it inside a loop.
2. **Being polite to APIs**: pause between requests with `time.sleep()`.
3. **A summary report**: loop over the files you downloaded and print the key facts about each one.
4. **Save only what you need**: turn the data into Python, keep the parts you use, and save a smaller file.

This follows on from **DS5_1** (loops and conditionals) and **DS5_2** (downloading FRED data).

> **Before you start:** click **File → Save a copy in Drive**. Otherwise, your changes will not be saved.

In [ ]:
import requests   # fetch things from the internet
import json       # read JSON files
import time       # pause (and other things to do with time)

<br><br>

---

<br>

## 1. Building a list

Often we want a loop to **collect** results, to use later (in a chart, a table or a file). The pattern:

1. Make an **empty list**, `[]`, *before* the loop.
2. Inside the loop, add to it with **`.append()`**.

Here we collect only the inflation rates above the 2% target:

In [ ]:
inflation_rates = [3.8, 1.7, 2.0, 5.2, 2.6]
above_target = []          # an empty list, to fill up

for rate in inflation_rates:
    if rate > 2:
        above_target.append(rate)      # add this rate to the end of the list
        print(f"{rate} is above target, so we add it. The list is now: {above_target}")
    else:
        print(f"{rate} is not above target, so we skip it")

print()
print(f"Above target: {above_target}")
print(f"That's {len(above_target)} out of {len(inflation_rates)}")

<br>

> **Why before the loop?** If `above_target = []` were *inside* the loop, the list would be emptied again every time round, and we'd end up with at most one item.

A second example: build a list of web links, one for each series code.

In [ ]:
series_codes = ["UNRATE", "CPIAUCSL", "GDPC1"]
links = []

for series_id in series_codes:
    links.append(f"https://fred.stlouisfed.org/series/{series_id}")

links

<br><br>

---

<br>

## 2. Being polite to APIs: `time.sleep()`

Every request asks someone else's server to do some work. If we send lots of requests very quickly (easy with a loop!), the server may:
- reply with error **429**, which means *Too Many Requests*,
- or block us for a while.

Most APIs set a limit. FRED allows **120 requests a minute**, and the [API documentation](https://fred.stlouisfed.org/docs/api/fred/) of each site tells you its rules. When scraping websites (Week 4), it's even more important to slow down.

The fix is simple: **pause** between requests. `time.sleep(1)` waits one second. Run this and watch the lines appear one at a time:

In [ ]:
for number in [1, 2, 3]:
    print(f"Request {number}... then wait 1 second")
    time.sleep(1)

print("Done")

<br>

Now add a pause to the download loop from DS5_2. Put your series codes in the list (the same ones as in DS5_2 is fine):

In [ ]:
api_key = "22ee7a76e736e32f54f5df0a7171538d"   # the shared course key, or your own (see DS5_2)
fred_series = []      # your series codes, e.g. ["UNRATE", "CPIAUCSL", ...]

for series_id in fred_series:
    print(f"--- Series: {series_id} ---")

    url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json"
    response = requests.get(url)

    if response.status_code == 200:
        filename = f"data_US_FRED-{series_id}.json"
        with open(filename, "w", encoding="utf-8") as file:
            file.write(response.text)
        print(f"OK: saved as {filename}")
    else:
        print(f"Problem with {series_id}: error {response.status_code}")

    time.sleep(0.5)   # wait half a second before the next request
    print()

print("Finished")

<br>

> **How long should I wait?** Half a second to one second is fine for most APIs. If you get error 429, wait longer (or check the API's rules). For a few requests it makes little difference; for hundreds, it's what keeps your code working.

<br><br>

---

<br>

## 3. A summary report

We now have several files. What's in them? A loop can tell us, without opening each one by hand.

As always, start with **one** file. `fred_series[0]` is the first code in your list:

In [ ]:
filename = f"data_US_FRED-{fred_series[0]}.json"

with open(filename, encoding="utf-8") as file:      # open the file to read it
    data = json.load(file)        # turn the JSON text into a Python dictionary

observations = data["observations"]
print(f"{filename} has {len(observations)} observations")

observations[-1]     # the last observation ([-1] counts from the end)

<br>

We want the **latest value**. But FRED writes missing values as `"."`, so the last observation may be empty.

So we loop over every observation, and **if** the value isn't missing:
- store it as `latest`. This is overwritten each time, so when the loop ends it holds the **last** real value.
- add it (as a number) to a list, `values` (pattern 1!), so we can find the highest and lowest.

In [ ]:
latest = None     # nothing yet
values = []       # an empty list

for observation in observations:
    if observation["value"] != ".":                    # skip missing values
        latest = observation                           # replaced each time round
        values.append(float(observation["value"]))     # float() turns the text '3.4' into the number 3.4

print(f"Latest: {latest['value']} ({latest['date']})")
print(f"Highest: {max(values)}, lowest: {min(values)}")

<br>

Now the same steps for **every** file. This is a loop *inside* a loop:
- the **outer** loop goes through the series, one file at a time,
- the **inner** loop goes through the observations in that file.

In [ ]:
print("FRED data summary")
print("=================")

for series_id in fred_series:
    with open(f"data_US_FRED-{series_id}.json", encoding="utf-8") as file:
        observations = json.load(file)["observations"]

    latest = None
    values = []
    for observation in observations:
        if observation["value"] != ".":
            latest = observation
            values.append(float(observation["value"]))

    print(f"{series_id}: latest {latest['value']} ({latest['date']}). "
          f"Highest {max(values)}, lowest {min(values)}. Data from {observations[0]['date']}")

<br>

### Save the report

Instead of printing, we can **build a list** of results: one dictionary per series. Saved as JSON, this could become the data for a table or chart on your website.

In [ ]:
summary = []

for series_id in fred_series:
    with open(f"data_US_FRED-{series_id}.json", encoding="utf-8") as file:
        observations = json.load(file)["observations"]

    latest = None
    values = []
    for observation in observations:
        if observation["value"] != ".":
            latest = observation
            values.append(float(observation["value"]))

    summary.append({
        "series": series_id,
        "latest_date": latest["date"],
        "latest_value": float(latest["value"]),
        "highest": max(values),
        "lowest": min(values),
    })

with open("summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)     # indent=2 makes the file easy to read

summary

<br><br>

---

<br>

## 4. Save only what you need

So far we've saved exactly what FRED sends, with `file.write(response.text)`. That's simple, and it keeps the original data.

But look at one observation: there are **4 fields**, and our charts only use 2 (`date` and `value`). The other two are repeated on every row. If we want a smaller, tidier file, we can:
1. **Parse**: turn the text into a Python dictionary, with `response.json()`.
2. **Keep** only what we need, with a loop (pattern 1: building a list).
3. **Write**: turn it back into JSON text and save it, with `json.dump()`.

As always, one series first:

In [ ]:
series_id = fred_series[0]
url = f"https://api.stlouisfed.org/fred/series/observations?series_id={series_id}&api_key={api_key}&file_type=json"
response = requests.get(url)

data = response.json()            # 1. parse: JSON text -> Python dictionary
data["observations"][0]           # one observation: 4 fields

In [ ]:
small = []                        # 2. keep only the date and value of each observation
for observation in data["observations"]:
    small.append({"date": observation["date"], "value": observation["value"]})

small_filename = f"data_US_FRED-{series_id}_small.json"
with open(small_filename, "w", encoding="utf-8") as file:
    json.dump(small, file)        # 3. write: Python list -> JSON text in a file

small[:3]

<br>

How much smaller is it? `os.path.getsize()` gives a file's size in bytes:

In [ ]:
import os

full_size = os.path.getsize(f"data_US_FRED-{series_id}.json")    # the full file from section 2
small_size = os.path.getsize(small_filename)

print(f"Full file:  {full_size:,} bytes")
print(f"Small file: {small_size:,} bytes ({small_size / full_size:.0%} of the full size)")

<br>

One more difference: the small file is just a **list**, with no `observations` wrapper. So a Vega-Lite chart using it doesn't need `"property"`:

```json
"data": {"url": "data_US_FRED-UNRATE_small.json"}
```

> **Which should I use?** Saving `response.text` is simplest and keeps everything, so it's a good default. Parse and `json.dump()` when you want to change the data before saving: keep fewer fields, fewer years, or fix values.

<br><br>

---

<br>

## <font color='green'>Try it</font>

**1.** In Week 4 we downloaded World Bank data for one country. Write a loop that downloads it for five countries, waiting one second between requests, and builds a list of the countries that worked.

**2.** Add the **date** of the highest value to the report. Hint: inside the inner loop, keep a variable `highest` and use an `if` to replace it whenever you find a bigger value.

**3.** Add an alert: print `⚠️ up on a year ago` if the latest value is higher than 12 observations earlier (for a monthly series). Hint: `values[-1]` is the latest value and `values[-13]` the one 12 before it.

**4.** Change section 4 so the small file skips missing values (`"."`) and stores each value as a number, with `float()`. Then turn it into a loop that makes a small file for every series in your list.

In [ ]:
# Your code here
